# ACME Expenses - Bronze Tier

This AIDP notebook creates the Bronze schema and Bronze tables for the ACME Expenses medallion use case.

The Bronze layer keeps a direct copy of source expense tables, then creates lightly cleaned Bronze-ready tables for downstream Silver processing and ML feature preparation.

In this notebook we:

1. Create the `bronze` schema in the `acme_expenses` catalog.
2. Copy `acme_expenses.expenses.acme_expense_review_training` into `acme_expenses.bronze.expenses_training_bronze`.
3. Copy `acme_expenses.expenses.acme_expenses` into `acme_expenses.bronze.expenses_bronze`.
4. Show examples of syntax using SQL and PySpark.
5. Remove rows that are not useful for a model that predicts whether an expense should be reviewed.
6. Use an AIDP foundational LLM through `query_model(...)` to enrich a dataset.
        

In [38]:
# AIDP notebook configuration
catalog_name = "acme_expenses"
source_schema = "expenses"
bronze_schema = "bronze"

source_training_table_name = "acme_expense_review_training"
source_expenses_table_name = "acme_expenses"

training_bronze_table_name = "expenses_training_bronze"
expenses_bronze_table_name = "expenses_bronze"

training_clean_table_name = "expenses_training_bronze_clean"
expenses_clean_table_name = "expenses_bronze_clean"
llm_table_name = "expenses_training_bronze_llm_enriched_sample"

llm_model_name = "default.oci_ai_models.cohere.command-a-03-2025"
        

In [39]:
from functools import reduce

from pyspark.sql import functions as F
from pyspark.sql.window import Window


def quote_ident(name: str) -> str:
    return ".".join(f"`{part}`" for part in name.split("."))


def table_path(schema_name: str, table_name: str) -> str:
    return f"{catalog_name}.{schema_name}.{table_name}"


def quoted_table_path(schema_name: str, table_name: str) -> str:
    return quote_ident(table_path(schema_name, table_name))


def run_sql(label: str, statement: str):
    print(f"\n-- {label}")
    print(statement)
    return spark.sql(statement)


def display_or_show(df, rows: int = 20):
    # AIDP notebooks support rich display in many workspaces. show() is the portable fallback.
    try:
        display(df.limit(rows))
    except NameError:
        df.show(rows, truncate=False)
    except Exception:
        df.show(rows, truncate=False)


source_training_table = table_path(source_schema, source_training_table_name)
source_expenses_table = table_path(source_schema, source_expenses_table_name)

training_bronze_table = table_path(bronze_schema, training_bronze_table_name)
expenses_bronze_table = table_path(bronze_schema, expenses_bronze_table_name)
training_clean_table = table_path(bronze_schema, training_clean_table_name)
expenses_clean_table = table_path(bronze_schema, expenses_clean_table_name)
llm_table = table_path(bronze_schema, llm_table_name)
        

## 1. Create the Bronze Schema

This creates `acme_expenses.bronze` using AIDP/Spark SQL DDL.
        

In [3]:
run_sql(
    "create bronze schema",
    f"CREATE SCHEMA IF NOT EXISTS {quote_ident(catalog_name)}.{quote_ident(bronze_schema)}"
)

spark.sql(f"SHOW SCHEMAS IN {quote_ident(catalog_name)}").show(truncate=False)
        


-- create bronze schema
CREATE SCHEMA IF NOT EXISTS `acme_expenses`.`bronze`


+---------+
|namespace|
+---------+
|bronze   |
|expenses |
|default  |
+---------+



## 2. Copy Source Tables into Bronze

These Bronze tables are direct table copies from the `expenses` schema. The notebook drops and recreates them each time.
        

In [4]:
table_copies = [
    {
        "source": source_training_table,
        "target": training_bronze_table,
        "description": "Training view with review labels and engineered expense features",
    },
    {
        "source": source_expenses_table,
        "target": expenses_bronze_table,
        "description": "Newer expenses to score for review routing",
    },
]

for spec in table_copies:
    run_sql("drop existing target", f"DROP TABLE IF EXISTS {quote_ident(spec['target'])}")
    run_sql(
        "copy source table into bronze",
        f"CREATE TABLE {quote_ident(spec['target'])} AS SELECT * FROM {quote_ident(spec['source'])}"
    )

copy_audit_rows = []
for spec in table_copies:
    row_count = spark.read.table(spec["target"]).count()
    copy_audit_rows.append(
        {
            "source_table": spec["source"],
            "target_table": spec["target"],
            "description": spec["description"],
            "row_count": row_count,
        }
    )

copy_audit_df = spark.createDataFrame(copy_audit_rows)
display_or_show(copy_audit_df, 10)
        


-- drop existing target
DROP TABLE IF EXISTS `acme_expenses`.`bronze`.`expenses_training_bronze`

-- copy source table into bronze
CREATE TABLE `acme_expenses`.`bronze`.`expenses_training_bronze` AS SELECT * FROM `acme_expenses`.`expenses`.`acme_expense_review_training`



-- drop existing target
DROP TABLE IF EXISTS `acme_expenses`.`bronze`.`expenses_bronze`



-- copy source table into bronze
CREATE TABLE `acme_expenses`.`bronze`.`expenses_bronze` AS SELECT * FROM `acme_expenses`.`expenses`.`acme_expenses`


## 3. Display Bronze Tables with PySpark
        

In [5]:
training_bronze_df = spark.read.table(training_bronze_table)
expenses_bronze_df = spark.read.table(expenses_bronze_table)

print(f"{training_bronze_table}: {training_bronze_df.count()} rows, {len(training_bronze_df.columns)} columns")
display_or_show(training_bronze_df, 20)

print(f"{expenses_bronze_table}: {expenses_bronze_df.count()} rows, {len(expenses_bronze_df.columns)} columns")
display_or_show(expenses_bronze_df, 20)
        

acme_expenses.bronze.expenses_training_bronze: 2500 rows, 37 columns


acme_expenses.bronze.expenses_bronze: 2500 rows, 39 columns


## 4. Display Bronze Tables with SQL
        

In [6]:
%sql
SELECT *
FROM acme_expenses.bronze.expenses_training_bronze
LIMIT 20
        

In [7]:
%sql
SELECT *
FROM acme_expenses.bronze.expenses_bronze
LIMIT 20
        

## 5. Clean Rows for ML Readiness

For the future model that predicts whether an expense should be reviewed, this cleaning step removes rows that are not useful for training or scoring:

- Missing `claim_id`
- Missing `expense_date`, `expense_type`, or `amount`
- Non-positive `amount`
- Duplicate `claim_id` rows, keeping the latest expense date

The output stays in Bronze as a lightly cleaned landing table. The silver notebook will use this data for model creation.
        

In [24]:
from functools import reduce
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog_name = globals().get("catalog_name", "acme_expenses")
bronze_schema = globals().get("bronze_schema", "bronze")

training_bronze_table = globals().get(
    "training_bronze_table",
    f"{catalog_name}.{bronze_schema}.expenses_training_bronze"
)
expenses_bronze_table = globals().get(
    "expenses_bronze_table",
    f"{catalog_name}.{bronze_schema}.expenses_bronze"
)
training_clean_table = globals().get(
    "training_clean_table",
    f"{catalog_name}.{bronze_schema}.expenses_training_bronze_clean"
)
expenses_clean_table = globals().get(
    "expenses_clean_table",
    f"{catalog_name}.{bronze_schema}.expenses_bronze_clean"
)

def quote_ident(name):
    return ".".join(f"`{part}`" for part in name.split("."))

def actual_column(df, name):
    lookup = {c.lower(): c for c in df.columns}
    return lookup.get(name.lower())

def first_existing_column(df, candidates):
    lookup = {c.lower(): c for c in df.columns}
    return next((lookup[c.lower()] for c in candidates if c.lower() in lookup), None)

def normalize_amount(col_name):
    return F.regexp_replace(F.col(col_name).cast("string"), r"[^0-9.\-]", "").cast("double")

def normalize_review_label(col_name):
    raw = F.lower(F.trim(F.col(col_name).cast("string")))
    return (
        F.when(raw.isin("1", "y", "yes", "true", "review", "review required", "required", "needs review"), 1)
         .when(raw.isin("0", "n", "no", "false", "approved", "not required", "not reviewed", "none"), 0)
         .otherwise(F.col(col_name).cast("int"))
    )

def normalize_expense_type(col_name):
    raw = F.lower(F.trim(F.col(col_name).cast("string")))
    return (
        F.when(raw.isin("meal", "meals"), "Meals")
         .when(raw.isin("lodging", "hotel", "hotels"), "Lodging")
         .when(raw.isin("transport", "transportation", "ground transportation", "travel"), "Transportation")
         .otherwise(F.trim(F.col(col_name).cast("string")))
    )

def dedupe_by_claim_id(df):
    claim_col = actual_column(df, "claim_id")
    if not claim_col:
        return df

    date_col = actual_column(df, "expense_date")
    order_cols = [F.col(date_col).desc_nulls_last()] if date_col else []
    order_cols.append(F.monotonically_increasing_id().desc())

    w = Window.partitionBy(claim_col).orderBy(*order_cols)
    return df.withColumn("_rn", F.row_number().over(w)).filter(F.col("_rn") == 1).drop("_rn")

training_bronze_df = spark.read.table(training_bronze_table)
expenses_bronze_df = spark.read.table(expenses_bronze_table)

target_candidates = [
    "training_label_review_required",
    "review_required",
    "review_required_flag",
    "needs_review",
    "needs_review_flag",
    "review_flag",
    "requires_review",
]

raw_target_col = first_existing_column(training_bronze_df, target_candidates)
if raw_target_col is None:
    raise ValueError(f"No review target column found. Checked: {target_candidates}")

def prepare_common_columns(df, include_target=False):
    out = df

    amount_col = actual_column(out, "amount")
    if amount_col:
        out = out.withColumn(amount_col, normalize_amount(amount_col))

    expense_date_col = actual_column(out, "expense_date")
    if expense_date_col:
        out = out.withColumn(expense_date_col, F.to_date(F.col(expense_date_col)))

    submission_date_col = actual_column(out, "submission_date")
    if submission_date_col:
        out = out.withColumn(submission_date_col, F.to_date(F.col(submission_date_col)))

    expense_type_col = actual_column(out, "expense_type")
    if expense_type_col:
        out = out.withColumn(expense_type_col, normalize_expense_type(expense_type_col))

    if include_target:
        out = out.withColumn("_bronze_review_required_label", normalize_review_label(raw_target_col))

    return out

def ml_ready_filter(df, require_target=False):
    filters = []

    for col_name in ["claim_id", "expense_date", "expense_type", "amount"]:
        c = actual_column(df, col_name)
        if c:
            filters.append(F.col(c).isNotNull())

    employee_col = first_existing_column(df, ["canonical_employee_id", "employee_id", "employee_email"])
    if employee_col:
        filters.append(F.col(employee_col).isNotNull())

    amount_col = actual_column(df, "amount")
    if amount_col:
        filters.append(F.col(amount_col) > 0)

    expense_type_col = actual_column(df, "expense_type")
    if expense_type_col:
        filters.append(F.col(expense_type_col).isin("Meals", "Lodging", "Transportation"))

    if require_target:
        filters.append(F.col("_bronze_review_required_label").isin(0, 1))

    return reduce(lambda a, b: a & b, filters)

training_prepared_df = prepare_common_columns(training_bronze_df, include_target=True)
expenses_prepared_df = prepare_common_columns(expenses_bronze_df, include_target=False)

training_clean_df = (
    dedupe_by_claim_id(training_prepared_df.filter(ml_ready_filter(training_prepared_df, require_target=True)))
    .withColumn("_bronze_cleaned_at", F.current_timestamp())
)

expenses_clean_df = (
    dedupe_by_claim_id(expenses_prepared_df.filter(ml_ready_filter(expenses_prepared_df, require_target=False)))
    .withColumn("_bronze_cleaned_at", F.current_timestamp())
)

spark.sql(f"DROP TABLE IF EXISTS {quote_ident(training_clean_table)}")
training_clean_df.write.mode("overwrite").saveAsTable(training_clean_table)

spark.sql(f"DROP TABLE IF EXISTS {quote_ident(expenses_clean_table)}")
expenses_clean_df.write.mode("overwrite").saveAsTable(expenses_clean_table)

quality_summary = spark.createDataFrame([
    {
        "table_name": training_clean_table,
        "raw_rows": training_bronze_df.count(),
        "clean_rows": training_clean_df.count(),
        "rows_removed": training_bronze_df.count() - training_clean_df.count(),
        "raw_target_column": raw_target_col,
        "normalized_target_column": "_bronze_review_required_label",
    },
    {
        "table_name": expenses_clean_table,
        "raw_rows": expenses_bronze_df.count(),
        "clean_rows": expenses_clean_df.count(),
        "rows_removed": expenses_bronze_df.count() - expenses_clean_df.count(),
        "raw_target_column": None,
        "normalized_target_column": None,
    },
])

quality_summary.show(truncate=False)
        

+----------+-----------------------------+--------+------------------------------+------------+---------------------------------------------------+
|clean_rows|normalized_target_column     |raw_rows|raw_target_column             |rows_removed|table_name                                         |
+----------+-----------------------------+--------+------------------------------+------------+---------------------------------------------------+
|2500      |_bronze_review_required_label|2500    |training_label_review_required|0           |acme_expenses.bronze.expenses_training_bronze_clean|
|2500      |NULL                         |2500    |NULL                          |0           |acme_expenses.bronze.expenses_bronze_clean         |
+----------+-----------------------------+--------+------------------------------+------------+---------------------------------------------------+



## 6. Display Cleaned Bronze Tables
        

In [25]:
%sql
SELECT *
FROM acme_expenses.bronze.expenses_training_bronze_clean
LIMIT 20
        

In [27]:
%sql
SELECT *
FROM acme_expenses.bronze.expenses_bronze_clean
LIMIT 20
        

## 7. Foundational LLM Enrichment in AIDP

AIDP exposes foundation models through `query_model(...)`. This example uses a model to turn structured policy signals into a concise standardized review reason.

The cell creates a sample enrichment table for rows likely to be relevant to review routing. To enrich the full cleaned training table, remove the `LIMIT 100` clause.
        

In [29]:
%sql
SELECT
  claim_id,
  expense_type,
  amount,
  receipt_attached_flag,
  days_to_submit,
  policy_exception_flag,
  query_model(
    "default.oci_ai_models.google.gemini-2.5-pro",
    CONCAT(
      "Return one concise expense compliance review reason. Use a short phrase only. ",
      "Fields: expense_type=", COALESCE(CAST(expense_type AS STRING), "unknown"),
      "; amount=", COALESCE(CAST(amount AS STRING), "unknown"),
      "; receipt_attached_flag=", COALESCE(CAST(receipt_attached_flag AS STRING), "unknown"),
      "; days_to_submit=", COALESCE(CAST(days_to_submit AS STRING), "unknown"),
      "; policy_exception_flag=", COALESCE(CAST(policy_exception_flag AS STRING), "unknown")
    )
  ) AS llm_policy_review_reason
FROM acme_expenses.bronze.expenses_training_bronze_clean
WHERE COALESCE(CAST(policy_exception_flag AS INT), 0) = 1
LIMIT 10
        

In [31]:
%sql
DROP TABLE IF EXISTS acme_expenses.bronze.expenses_training_bronze_llm_enriched_sample
        

OK

In [32]:
%sql
CREATE TABLE acme_expenses.bronze.expenses_training_bronze_llm_enriched_sample AS
SELECT
  sample_rows.*,
  query_model(
    "default.oci_ai_models.google.gemini-2.5-pro",
    CONCAT(
      "Return one concise expense compliance review reason. Use a short phrase only. ",
      "Fields: expense_type=", COALESCE(CAST(expense_type AS STRING), "unknown"),
      "; amount=", COALESCE(CAST(amount AS STRING), "unknown"),
      "; receipt_attached_flag=", COALESCE(CAST(receipt_attached_flag AS STRING), "unknown"),
      "; days_to_submit=", COALESCE(CAST(days_to_submit AS STRING), "unknown"),
      "; policy_exception_flag=", COALESCE(CAST(policy_exception_flag AS STRING), "unknown")
    )
  ) AS llm_policy_review_reason
FROM (
  SELECT *
  FROM acme_expenses.bronze.expenses_training_bronze_clean
  WHERE COALESCE(CAST(policy_exception_flag AS INT), 0) = 1
  LIMIT 100
) sample_rows
        

OK

## 8. Display the LLM-Enriched Sample
        

In [33]:
%sql
SELECT
  claim_id,
  expense_type,
  amount,
  policy_exception_flag,
  llm_policy_review_reason
FROM acme_expenses.bronze.expenses_training_bronze_llm_enriched_sample
LIMIT 20
        

In [40]:
llm_enriched_df = spark.read.table("acme_expenses.bronze.expenses_training_bronze_llm_enriched_sample")
display_or_show(
    llm_enriched_df.select(
        "claim_id",
        "expense_type",
        "amount",
        "policy_exception_flag",
        "llm_policy_review_reason",
    ),
    20,
)
        

## 9. Bronze Output Tables

This Bronze notebook creates these tables:

| Table | Purpose |
|---|---|
| `acme_expenses.bronze.expenses_training_bronze` | Direct copy of source training table |
| `acme_expenses.bronze.expenses_bronze` | Direct copy of source scoring expenses table |
| `acme_expenses.bronze.expenses_training_bronze_clean` | ML-ready cleaned training rows |
| `acme_expenses.bronze.expenses_bronze_clean` | ML-ready cleaned scoring rows |
| `acme_expenses.bronze.expenses_training_bronze_llm_enriched_sample` | Sample table with LLM-generated review reason enrichment |

        